# DA624 Submission Pipeline — Intent-Aware Learning

This notebook runs the emergency reduced-scale experiment on a Colab T4. It assumes the finished repository has already been pushed to GitHub. Run cells top-to-bottom. The pipeline trains three LoRA adapters: NTP baseline, concept loss, and the context-preserving extension.


In [ ]:
!nvidia-smi

import os
REPO = "/content/Intent-Aware-Learning-DA624-NLP"
if not os.path.exists(REPO):
    !git clone https://github.com/ShivasheeshDubey/Intent-Aware-Learning-DA624-NLP.git $REPO
%cd $REPO


In [ ]:
%pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm
%pip install -q -e .


## 1. Sanity check (optional but useful)


In [ ]:
!python scripts/smoke_test_model.py


## 2. C4 content-word extraction


In [ ]:
!mkdir -p data/processed data/splits checkpoints results
!python scripts/prepare_data.py \
  --dataset c4 \
  --model Qwen/Qwen3-1.7B-Base \
  --max-samples 120 \
  --output data/processed/c4_content.jsonl


## 3. Top-k candidate generation with the base LM


In [ ]:
!python scripts/generate_candidates.py \
  --input data/processed/c4_content.jsonl \
  --output data/processed/c4_candidates.jsonl \
  --model Qwen/Qwen3-1.7B-Base \
  --max-targets-per-sequence 4 \
  --top-k 100 \
  --max-candidates 30


## 4. Contextual filtering with Qwen3-1.7B instruct/post-trained model


In [ ]:
!python scripts/filter_concepts.py \
  --input data/processed/c4_candidates.jsonl \
  --output data/processed/c4_concepts.jsonl \
  --model Qwen/Qwen3-1.7B \
  --batch-size 8 \
  --max-synonyms 10


## 5. Split C4 and prepare OpenWebText OOD evaluation data


In [ ]:
!python scripts/split_data.py \
  --input data/processed/c4_concepts.jsonl \
  --output-dir data/splits/c4

!python scripts/prepare_data.py \
  --dataset openwebtext \
  --model Qwen/Qwen3-1.7B-Base \
  --max-samples 60 \
  --output data/processed/openwebtext_eval.jsonl


## 6. Train NTP baseline (lambda=0)


In [ ]:
!python scripts/train_model.py \
  --train data/splits/c4/train.jsonl \
  --output-dir checkpoints/ntp \
  --model Qwen/Qwen3-1.7B-Base \
  --concept-weight 0.0 \
  --epochs 2


## 7. Train reproduced concept objective (lambda=0.5)


In [ ]:
!python scripts/train_model.py \
  --train data/splits/c4/train.jsonl \
  --output-dir checkpoints/concept \
  --model Qwen/Qwen3-1.7B-Base \
  --concept-weight 0.5 \
  --target-mass 0.6 \
  --epochs 2


## 8. Train our context-preserving extension


In [ ]:
!python scripts/train_model.py \
  --train data/splits/c4/train.jsonl \
  --output-dir checkpoints/ours \
  --model Qwen/Qwen3-1.7B-Base \
  --concept-weight 0.5 \
  --target-mass 0.6 \
  --preserve-beta 0.1 \
  --epochs 2


## 9. In-domain C4 evaluation


In [ ]:
!python scripts/evaluate_model.py --data data/splits/c4/test.jsonl --output results/c4_base.json --base-model Qwen/Qwen3-1.7B-Base
!python scripts/evaluate_model.py --data data/splits/c4/test.jsonl --output results/c4_ntp.json --base-model Qwen/Qwen3-1.7B-Base --adapter checkpoints/ntp
!python scripts/evaluate_model.py --data data/splits/c4/test.jsonl --output results/c4_concept.json --base-model Qwen/Qwen3-1.7B-Base --adapter checkpoints/concept
!python scripts/evaluate_model.py --data data/splits/c4/test.jsonl --output results/c4_ours.json --base-model Qwen/Qwen3-1.7B-Base --adapter checkpoints/ours


## 10. OOD OpenWebText evaluation


In [ ]:
!python scripts/evaluate_model.py --data data/processed/openwebtext_eval.jsonl --output results/owt_base.json --base-model Qwen/Qwen3-1.7B-Base
!python scripts/evaluate_model.py --data data/processed/openwebtext_eval.jsonl --output results/owt_concept.json --base-model Qwen/Qwen3-1.7B-Base --adapter checkpoints/concept
!python scripts/evaluate_model.py --data data/processed/openwebtext_eval.jsonl --output results/owt_ours.json --base-model Qwen/Qwen3-1.7B-Base --adapter checkpoints/ours


## 11. Paired bootstrap confidence intervals


In [ ]:
!python scripts/compare_results.py \
  --base results/c4_base.json \
  --concept results/c4_concept.json \
  --output results/bootstrap_concept_vs_base.json


## 12. Human semantic-similarity benchmarks

For the emergency run, each benchmark is capped at 300 pairs. Remove `--max-pairs 300` for the full MEN, WordSim353, SimLex999, and STS-B evaluations.


In [ ]:
!python scripts/download_benchmarks.py --output-dir data/benchmarks
!python scripts/evaluate_similarity.py --benchmark-dir data/benchmarks --output results/semantic_base.json --base-model Qwen/Qwen3-1.7B-Base --max-pairs 300
!python scripts/evaluate_similarity.py --benchmark-dir data/benchmarks --output results/semantic_concept.json --base-model Qwen/Qwen3-1.7B-Base --adapter checkpoints/concept --max-pairs 300
!python scripts/evaluate_similarity.py --benchmark-dir data/benchmarks --output results/semantic_ours.json --base-model Qwen/Qwen3-1.7B-Base --adapter checkpoints/ours --max-pairs 300


## 13. Generate the final Markdown report from actual results


In [ ]:
!python scripts/render_report.py

from pathlib import Path
print(Path("SUBMISSION_REPORT.md").read_text()[:8000])


## 14. Save results back to GitHub

Colab cannot push unless you authenticate Git. Download `SUBMISSION_REPORT.md` and the `results/` JSON files or commit/push them using your own GitHub credentials. Never add `checkpoints/` to GitHub.
